In [1]:
import time
from typing import List, Optional

import polars as pl
from ddgs import DDGS

In [2]:
def search_with_urls(
    query: str, max_results: int = 5, delay_seconds: float = 1.0, max_retries: int = 2
) -> tuple[List[Optional[str]], List[Optional[str]]]:
    """
    Search DuckDuckGo and return both text snippets and URLs.

    Args:
        query: The search query
        max_results: Number of results to retrieve (default: 5)
        delay_seconds: Delay between searches to avoid rate limiting
        max_retries: Number of retries if search fails

    Returns:
        Tuple of (list of text snippets, list of URLs)
    """
    texts = []
    urls = []

    for attempt in range(max_retries):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=max_results))

                for result in results:
                    texts.append(result.get("body", "No text available"))
                    urls.append(result.get("href", "No URL available"))

                # If we got fewer results than requested, pad with None
                while len(texts) < max_results:
                    texts.append(None)
                    urls.append(None)

                time.sleep(delay_seconds)
                return texts, urls

        except Exception:
            if attempt < max_retries - 1:
                time.sleep(delay_seconds * 2)
                continue
            # Return all None values on complete failure
            return [None] * max_results, [None] * max_results

    return [None] * max_results, [None] * max_results


def process_column_with_searches(
    file_path: str,
    column_name: str,
    query_template: str = "Foundational year of {value}",
    max_results: int = 5,
    delay_seconds: float = 1.0,
    max_retries: int = 2,
) -> pl.DataFrame:
    """
    Read a CSV file and process each cell with parameterized searches.

    Args:
        file_path: Path to the CSV file
        column_name: Name of the column to process
        query_template: Template for the search query with {value} placeholder
        max_results: Number of search results per query (creates 2*max_results columns)
        delay_seconds: Delay between searches to avoid rate limiting
        max_retries: Maximum retry attempts per search

    Returns:
        Polars DataFrame with original data and added search result columns
    """
    # Read the CSV file
    df = pl.read_csv(file_path)

    # Prepare column names for results
    text_columns = [f"result_{i + 1}_text" for i in range(max_results)]
    url_columns = [f"result_{i + 1}_url" for i in range(max_results)]
    all_new_columns = text_columns + url_columns

    # Initialize lists to store results for each row
    rows_data = []

    # Process each row
    for row_idx, row in enumerate(df.iter_rows(named=True)):
        cell_value = row[column_name]

        # Handle empty or null values
        if cell_value is None or cell_value == "":
            row_data = {**row}
            for col in all_new_columns:
                row_data[col] = "Empty or null value"
            rows_data.append(row_data)
            continue

        # Build the query
        query = query_template.format(value=cell_value)
        print(f"Processing row {row_idx + 1}: {query}")

        # Perform the search
        texts, urls = search_with_urls(
            query=query,
            max_results=max_results,
            delay_seconds=delay_seconds,
            max_retries=max_retries,
        )

        # Create row data with new columns
        row_data = {**row}

        # Add text results
        for i, text in enumerate(texts):
            if i < max_results:
                row_data[text_columns[i]] = text if text is not None else "No result"

        # Add URL results
        for i, url in enumerate(urls):
            if i < max_results:
                row_data[url_columns[i]] = url if url is not None else "No URL"

        rows_data.append(row_data)

    # Convert back to DataFrame
    result_df = pl.DataFrame(rows_data)

    return result_df


In [3]:
result = process_column_with_searches(
    file_path="petites_cites_de_caractere.csv",
    column_name="city",
    query_template="{value}, date d'homologation Petite Cité de Caractère",
    max_results=5,
    delay_seconds=0.65,
)


Processing row 1: lunas, date d'homologation Petite Cité de Caractère
Processing row 2: marvejols, date d'homologation Petite Cité de Caractère
Processing row 3: la malène, date d'homologation Petite Cité de Caractère
Processing row 4: montsaugeon, date d'homologation Petite Cité de Caractère
Processing row 5: joinville, date d'homologation Petite Cité de Caractère
Processing row 6: vertus, date d'homologation Petite Cité de Caractère
Processing row 7: vaucouleurs, date d'homologation Petite Cité de Caractère
Processing row 8: cormicy, date d'homologation Petite Cité de Caractère
Processing row 9: sainte-ménehould, date d'homologation Petite Cité de Caractère
Processing row 10: sézanne, date d'homologation Petite Cité de Caractère
Processing row 11: aÿ, date d'homologation Petite Cité de Caractère
Processing row 12: mouzon, date d'homologation Petite Cité de Caractère
Processing row 13: rocroi, date d'homologation Petite Cité de Caractère
Processing row 14: bar-sur-seine, date d'homolo

In [5]:
result.write_csv("./PCC_5_reults.csv")